# Neural Network (MLP) — Water Quality Targets

Trains a **multilayer perceptron** regression model for each of thirteen
water-quality target variables using the terminal modeling table
`data/final/epa-full.csv`, then evaluates every model on a held-out
test split and reports **R²**, **RMSE**, and **Error Rate** (symmetric MAPE, %).

This is the fourth model family. The targets, features, split, transform
bake-off and metrics are **identical** to `multiple_linear_regression.ipynb`,
`random_forest.ipynb` and `gradient_boosting.ipynb`, so all four are directly
comparable target by target.

**Targets modeled:** water temperature, dissolved oxygen, pH, nitrate, nitrite,
nitrate + nitrite, total phosphorus, specific conductance, total dissolved
solids, total suspended solids, turbidity, *E. coli*, and the composite WQI.

Each model is a scikit-learn `Pipeline`:

1. `SimpleImputer(strategy="median", add_indicator=True)` — fill missing
   predictor values **and append a binary was-missing column for each feature
   that had gaps in training**
2. `StandardScaler()` — zero mean, unit variance
3. `VotingRegressor` of three `MLPRegressor`s that differ only by seed

---

## Why this notebook differs from the tree families

Three things a forest tolerates and a network does not.

**Scaling is mandatory.** A random forest is invariant to monotone rescaling of
its inputs; gradient descent is not. `streamflow_discharge_cfs` and `awc_mean`
differ by five orders of magnitude, and without standardisation the first
feature dominates the initial gradient and the second never gets trained.
`StandardScaler` is the same step the linear-regression pipeline already uses.

**Missingness has to be encoded, not just filled.** Median imputation alone is
close to harmless for a tree — the tree can split around the imputed value and
isolate it. A network cannot: an imputed median is indistinguishable from a
genuinely median observation, so every gap silently becomes a confident
mid-range reading. Thirteen of the twenty-five base features carry NaNs, so
`add_indicator=True` appends a was-missing flag for each, letting the network
learn a different response for "measured and average" than for "not measured".
The pipeline still *accepts* the same 29 columns — the expansion is internal —
so `app.py` and the `feature_cols` contract are untouched.

**Early stopping has to be grouped by station.** `MLPRegressor(early_stopping=True)`
carves its validation slice off with a **random row split**, which puts the same
station on both sides and hands the network the exact leakage that
`GroupShuffleSplit` exists to remove — it would then stop training at whatever
epoch best memorises the training stations. This notebook therefore does its own
early stopping against a *station-grouped* inner validation split (see Part 1),
and sets `tol=0.0` with a large `n_iter_no_change` so scikit-learn's internal
stopping never fires and the epoch budget found here is what actually runs.

## Why three seeds

An MLP's score moves with its random initialisation far more than a forest's
does, and several test sets here are small — Nitrate + Nitrite is scored on
1,045 rows from 74 stations. A seed-to-seed swing on a set that size can be
larger than the entire margin the family is being judged on, so a single-seed
number would be noise reported as a result.

The saved model is therefore a `VotingRegressor` averaging three networks that
differ only by seed: it is both a steadier artifact and a stock scikit-learn
estimator, so the `.pkl` still contains no bespoke class to import at unpickle
time. **Part 3 reports the spread of the individual members** so the size of
that noise is visible rather than assumed.

---

**How the data is split.** `GroupShuffleSplit(test_size=0.2, random_state=42)`
grouped on `MonitoringLocationIdentifier`: 20% of the monitoring stations are
held out **whole**, so no station appears on both sides of the split and every
score below answers *"how well does this predict at a station the model has
never seen?"*. Because the split depends only on the group labels, it is the
same split the other three families use.

**Every R² is reported beside a persistence baseline** — "repeat this station's
previous value", a model with no features at all — measured on the same held-out
rows. The margin between them is how much the 29 predictors actually buy.

**Runtime** is a few minutes: the transform bake-off fits both arms across three
folds, each with its own epoch search.

In [1]:
# --- Imports ---
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import VotingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
RANDOM_STATE = 42
TEST_SIZE = 0.2               # share of *stations* held out — not share of rows
MIN_SAMPLES = 100             # skip a target with fewer usable rows than this
MIN_PERSISTENCE_PAIRS = 30    # below this the persistence baseline is not reported

# The train/test split is grouped on the station id: no monitoring location may
# appear on both sides. A plain random row split put 99%+ of test rows at a
# station that was also in training, and latitude/longitude are near-unique per
# station (~1 distinct value each per station) — so a model can recover the
# station from its coordinates and recall its typical level. That inflates every
# score. See src/04_eda/eda-summary.md, red flags 1-2.
GROUP_COL = "MonitoringLocationIdentifier"
DATE_COL = "ActivityStartDateTime"

# --- Network hyperparameters (shared across all targets) ------------------
# Deliberately modest. With 29 predictors and as few as 3,614 training rows on
# the smallest target, a wider network buys capacity this data cannot pay for;
# the constraint here is signal, not expressiveness.
#
# `tol=0.0` and the large `n_iter_no_change` disable scikit-learn's own
# convergence stopping. That is not a request to overfit — the epoch budget is
# chosen by the station-grouped search in Part 1 — it just makes `max_iter` mean
# exactly "run this many epochs" so the searched budget is the budget that runs.
MLP_PARAMS = dict(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-3,               # L2 penalty — the main brake on the small targets
    batch_size=256,
    learning_rate_init=1e-3,
    shuffle=True,
    early_stopping=False,     # replaced by the grouped search below
    tol=0.0,
    n_iter_no_change=10_000,
)

# The saved model averages these three networks; they differ only by seed.
SEEDS = (42, 43, 44)

# --- Station-grouped early stopping ---------------------------------------
INNER_VAL_SIZE = 0.15   # share of *training stations* held out to pick the epoch
MAX_EPOCHS = 300        # ceiling on the search
# Epochs without a validation-MSE improvement before stopping. 15 was measured
# first and is too impatient on the small targets: Nitrate + Nitrite has ~540
# inner-validation rows, its validation curve is flat and noisy between epochs 5
# and 32, and a patience of 15 latched onto the first dip and returned a
# 4-epoch network. The true validation optimum there is epoch 32.
PATIENCE = 25

## Configuration

Locate the dataset, and declare the targets and predictor features.

In [2]:
# --- Locate the repo root and the terminal modeling table ---
# The notebook may be launched from anywhere; walk upward until we find the CSV.
def find_data_path() -> Path:
    rel = Path("data/final/epa-full.csv")
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / rel
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {rel} by walking up from {here}. "
        "Run the notebook from within the repository."
    )

DATA_PATH = find_data_path()
print("Using dataset:", DATA_PATH)

# Repo root — where the .pkl files and model_metrics.csv are written.
REPO_ROOT = DATA_PATH.parents[2]
print("Repo root:", REPO_ROOT)

Using dataset: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/data/final/epa-full.csv
Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction


In [3]:
# --- Target variables: label -> (CSV column, plausible valid range) ---
# The valid range drops physically impossible readings and data-entry errors
# before fitting (e.g. a pH of 999 or a negative concentration).
TARGETS = {
    "Water Temperature":      {"column": "Temperature, water_value",            "valid_range": (-5.0, 45.0)},
    "Dissolved Oxygen":       {"column": "Dissolved oxygen (DO)_value",         "valid_range": (0.0, 30.0)},
    "pH":                     {"column": "pH_value",                            "valid_range": (0.0, 14.0)},
    "Nitrate":                {"column": "Nitrate_value",                       "valid_range": (0.0, 100.0)},
    "Nitrite":                {"column": "Nitrite_value",                       "valid_range": (0.0, 20.0)},
    "Nitrate + Nitrite":      {"column": "Nitrate + Nitrite_value",             "valid_range": (0.0, 100.0)},
    "Total Phosphorus":       {"column": "Total Phosphorus, mixed forms_value", "valid_range": (0.0, 25.0)},
    "Specific Conductance":   {"column": "Specific conductance_value",          "valid_range": (0.0, 10000.0)},
    "Total Dissolved Solids": {"column": "Total dissolved solids_value",        "valid_range": (0.0, 10000.0)},
    "Total Suspended Solids": {"column": "Total suspended solids_value",        "valid_range": (0.0, 10000.0)},
    "Turbidity":              {"column": "Turbidity_value",                     "valid_range": (0.0, 5000.0)},
    "E. coli":                {"column": "Escherichia coli_value",              "valid_range": (0.0, 1_000_000.0)},
    # Composite index added by src/04_eda/wqi-calculation.ipynb. 0 = best,
    # 100 = worst. It is a weighted roll-up of the other targets' measurements,
    # not an independent measurement -- but every water-quality `_value` column
    # is already excluded from FEATURE_COLS, so predicting it from environment
    # alone involves no leakage.
    #
    # Two caveats travel with it. It exists on only 56.6% of rows (943
    # stations), and `WQI_n_groups` -- how many of the eight pollution groups a
    # sample actually measured -- explains 9.7% of its variance on its own
    # (Spearman 0.25). Roughly a tenth of what a WQI model appears to learn is
    # therefore sampling design rather than water quality. That column is not a
    # feature, so the effect lands in the residual rather than being fitted;
    # it caps how well any of these models can do.
    "WQI":                    {"column": "WQI",                                "valid_range": (0.0, 100.0)},
}

# --- Predictor features ---
# Environmental / spatial / temporal drivers only. We deliberately exclude the
# other water-quality "_value" columns so a model never predicts one target
# from another measured target.
BASE_FEATURE_COLS = [
    # Location
    "LatitudeMeasure", "LongitudeMeasure",
    "distance_to_climate_station_km", "distance_to_streamflow_gauge_km",
    # PRISM climate normals at the observation
    "prism_tmax_c", "prism_tmin_c", "prism_ppt_mm", "prism_tdmean_c",
    # ISU station weather
    "isu_avg_wind_speed_kts", "isu_avg_rh", "isu_snow_in", "isu_snowd_in",
    "isu_max_feel_c", "isu_min_feel_c",
    # Hydrology
    "streamflow_discharge_cfs",
    # Soil
    "ksat_mean", "awc_mean",
    # Land cover. `pct_row_crops` is deliberately absent: it equals
    # pct_corn + pct_soybean exactly on all 48,251 rows, so including it made the
    # design matrix singular (rank 29 of 30, infinite condition number) without
    # adding a single fact. See src/04_eda/eda-summary.md §4.1.
    "pct_corn", "pct_soybean", "pct_developed", "pct_forest",
    # Nutrient loading context
    "npfert__n__total_kg", "npfert__p__total_kg",
    "npmanure__total__n_kg", "npmanure__total__p_kg",
]

# Temporal features engineered from the timestamp (added in the next cell).
TEMPORAL_FEATURE_COLS = ["doy", "doy_sin", "doy_cos", "obs_year"]

FEATURE_COLS = BASE_FEATURE_COLS + TEMPORAL_FEATURE_COLS
print(f"{len(FEATURE_COLS)} predictor features")

# --- Choosing the target scale -------------------------------------------
# Which targets get fitted on log10(y + c) is decided by measurement, not by a
# hand-written list: every eligible target is fitted BOTH ways and the winner
# is picked on a validation split carved out of the *training* stations. See
# `select_target_transform` in Part 1.
#
# Two guards decide which targets may enter that bake-off at all:
#
#   * a target with negative values has no log (Water Temperature);
#   * a target that is mostly zeros has no meaningful log scale. log10 maps the
#     whole point mass onto log10(c), and the resulting drop in MAE measures
#     how well the model predicts that mass, not how well it fits the water.
#     Nitrate (40% zeros) and Nitrite (85%) are hurdle-model problems, not
#     transform problems -- see src/04_eda/eda-summary.md 4.4.
MAX_ZERO_FRACTION_FOR_LOG = 0.20

# How the two arms are compared: 3-fold cross-validation over the *training*
# stations, averaged. A single hold-out split was measured and rejected — on
# the station-dominated targets one split is far too noisy.
N_SELECTION_FOLDS = 3

# The log arm must beat the raw arm by this *relative* margin on mean CV MAE.
# Ties go to the untransformed target: a transform is a real complication (a
# back-transform, a smearing correction, two scales to report), so it has to
# earn its place rather than win a coin flip.
MIN_LOG_MAE_GAIN = 0.05

29 predictor features


## Load and prepare the data

Parse the timestamp and derive seasonal (day-of-year) features.

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    # Derive temporal predictors from the activity timestamp. Build them as one
    # block and concat once, so we don't fragment the already-wide frame.
    ts = pd.to_datetime(df[DATE_COL], errors="coerce")
    doy = ts.dt.dayofyear
    radians = 2.0 * np.pi * doy / 365.25  # cyclical: day 365 sits next to day 1
    temporal = pd.DataFrame({
        "doy": doy,
        "obs_year": ts.dt.year,
        "doy_sin": np.sin(radians),
        "doy_cos": np.cos(radians),
        # Not a predictor — the ordering key for the persistence baseline.
        "_obs_ts": ts,
    }, index=df.index)
    return pd.concat([df, temporal], axis=1)


data = load_dataset(DATA_PATH)
print("Rows:", len(data), "| Columns:", data.shape[1])

# Sanity-check every declared predictor actually exists.
missing = [c for c in FEATURE_COLS if c not in data.columns]
assert not missing, f"Missing predictor columns: {missing}"

# How many features carry gaps — i.e. how many was-missing indicator columns
# the imputer will append inside the pipeline.
n_gappy = int(data[FEATURE_COLS].isna().any().sum())
print(f"{n_gappy} of {len(FEATURE_COLS)} predictors contain NaNs "
      "-> that many was-missing indicator columns are appended internally.")

data[FEATURE_COLS].describe().T[["count", "mean", "std", "min", "max"]]

Rows: 48251 | Columns: 323
13 of 29 predictors contain NaNs -> that many was-missing indicator columns are appended internally.


,count,mean,std,min,max
LatitudeMeasure,"48,251.0000",41.8428,0.7139,40.3871,43.5002
LongitudeMeasure,"48,251.0000",-93.0813,1.4052,-96.6325,-90.2010
distance_to_climate_station_km,"48,251.0000",21.5023,12.0707,0.2631,74.5673
distance_to_streamflow_gauge_km,"48,251.0000",5.0970,4.8408,0.0000,25.8260
prism_tmax_c,"46,602.0000",21.8390,9.8877,-20.4180,38.9453
prism_tmin_c,"46,632.0000",10.1485,9.2721,-29.3780,27.0850
prism_ppt_mm,"46,629.0000",3.3790,9.4365,0.0000,131.6350
prism_tdmean_c,"46,625.0000",10.8877,9.3835,-29.1303,27.2638
isu_avg_wind_speed_kts,"45,079.0000",7.0451,3.5216,0.0000,25.5105
isu_avg_rh,"44,806.0000",72.6710,12.8659,1.0227,100.0000


## Metrics

* **R²** — coefficient of determination on the held-out test set.
* **RMSE** — root mean squared error, in the target's own units.
* **Error Rate** — symmetric mean absolute percentage error (sMAPE), reported as
  a percentage. sMAPE is bounded and stays well-behaved when the true value is
  near zero, which matters for skewed concentration targets. This matches the
  `error_rate_pct` convention used elsewhere in the repo.

In [5]:
def symmetric_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric MAPE as a percentage (0 = perfect). Robust to y_true near 0."""
    denom = np.abs(y_true) + np.abs(y_pred)
    numer = 2.0 * np.abs(y_true - y_pred)
    safe = np.divide(numer, denom, out=np.zeros_like(denom, dtype=float), where=denom != 0)
    return float(np.mean(safe) * 100.0)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Error Rate (%)": symmetric_mape(y_true, y_pred),
    }


# --- The log10(y + c) target transform ------------------------------------
def log_offset(y_train: np.ndarray) -> float:
    """The c in log10(y + c): 1% of the positive training median.

    Chosen explicitly rather than left at a library default of 1. These targets
    live on very different units -- E. coli in MPN/100mL runs to 10^6, total
    phosphorus in mg/L rarely clears 1 -- so a fixed c = 1 would be a rounding
    error for one and wider than the entire distribution of the other. Fitted
    on the training rows only, so the test set never informs the transform.
    """
    positive = y_train[y_train > 0]
    if positive.size == 0:
        raise ValueError("no positive training values; cannot set a log offset")
    return float(0.01 * np.median(positive))


def to_log(y: np.ndarray, offset: float | None) -> np.ndarray:
    """Forward transform. `offset is None` means this target is fitted raw."""
    return y if offset is None else np.log10(y + offset)


def duan_smearing(resid_log: np.ndarray) -> float:
    """Duan's (1983) smearing estimate of the retransformation bias.

    E[y] is not 10 ** E[log10 y]: exponentiating a log-scale prediction
    under-estimates the mean by a factor that grows with the residual spread.
    Duan estimates that factor non-parametrically, as the mean of 10**residual
    over the *training* residuals. Skipping it biases every back-transformed
    prediction low.
    """
    return float(np.mean(10.0 ** resid_log))


def to_raw(y_log: np.ndarray, offset: float | None, smear: float = 1.0) -> np.ndarray:
    """Inverse transform, back to the target's own units.

    Clipped at zero: the log-fitted targets are concentrations whose valid range
    starts at 0, and subtracting the offset can otherwise push a very low
    prediction slightly negative.
    """
    if offset is None:
        return y_log
    return np.clip((10.0 ** y_log) * smear - offset, 0.0, None)


def predict_raw(entry: dict, X: np.ndarray) -> np.ndarray:
    """A trained model's prediction in the target's own units."""
    return to_raw(entry["model"].predict(X), entry["offset"], entry["smear"])

## Part 1 — Train every model

For each target we drop rows with no measurement, clip to the valid range, then
split **by station** with `GroupShuffleSplit(test_size=0.2)` grouped on
`MonitoringLocationIdentifier`: 20% of the *stations* that measured this target
are held out whole, and none of their rows are ever seen during training. The
row share of the test set therefore drifts away from 20% — station volume is
heavy-tailed (the median station has 7 observations, the busiest 2,878) — so the
actual row and station counts are printed per target.

### How many epochs

`search_epochs()` holds out 15% of the **training stations** as an inner
validation set, trains a single network one epoch at a time via `warm_start`,
and records the epoch at which validation **MSE** stops improving for `PATIENCE`
consecutive epochs. The winning epoch count is then used to refit from scratch
on the *whole* training set.

The inner split is grouped for the same reason the outer one is. Stopping
against a random slice of rows would choose the epoch that best memorises
stations the network has already seen — which is precisely the quantity the
outer grouped split was introduced to stop rewarding — and would push the budget
well past the point where genuine generalisation peaks.

**The criterion is MSE even though this repo reports and arbitrates on MAE**,
because within one arm it has to match the loss the network is descending. An
MAE criterion was measured first and is actively broken here: on raw-scale
Turbidity it stops at **epoch 1** where MSE stops at 114. Squared-error training
pulls the prediction off the median toward the mean, so on a right-skewed raw
target validation MAE rises from the first epoch while the fit is still
improving. The result was an untrained network at R² 0.007 — and because the
transform bake-off then compared two equally untrained arms, it could not see
the log arm's advantage and left Turbidity on the raw scale, which every other
family fits on the log. MAE stays the arbiter *between* arms, where the two
objectives differ and it is the only scale-neutral yardstick.

Two details make the budget mean what it says. `n_iter_` resets on every
`warm_start` call, so the loop counts epochs itself rather than reading it back
off the estimator; and `tol=0.0` with a large `n_iter_no_change` keeps
scikit-learn's own convergence check from halting the refit early.

The bake-off between the raw and log target scales runs the same search inside
each of its folds, so both arms are fitted under identical rules.

In [6]:
def make_preprocessor() -> Pipeline:
    """Impute (+ was-missing flags) -> standardise.

    Split out from `make_pipeline` so the epoch search can transform once and
    then train incrementally on arrays, instead of refitting the imputer and
    scaler on every one of up to 300 epochs.
    """
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])


def make_pipeline(max_iter: int, seeds: tuple[int, ...] = SEEDS) -> Pipeline:
    """Impute (+ flags) -> standardise -> seed-averaged MLP.

    With more than one seed the estimator is a `VotingRegressor`, which simply
    averages its members' predictions. It is a stock scikit-learn estimator, so
    the artifact still unpickles with no bespoke class to import.
    """
    members = [(f"mlp_{s}", MLPRegressor(**MLP_PARAMS, max_iter=max_iter, random_state=s))
               for s in seeds]
    estimator = members[0][1] if len(members) == 1 else VotingRegressor(members, n_jobs=-1)
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])


def search_epochs(X: np.ndarray, y: np.ndarray, groups: np.ndarray,
                  seed: int = RANDOM_STATE) -> tuple[int, float]:
    """Pick the number of training epochs on a station-grouped inner split.

    Trains one epoch at a time on 85% of the training *stations* and watches
    validation **MSE** on the held-out 15%. Returns (best_epoch, best_val_mse).

    MSE, not MAE, because the criterion has to match the objective the network
    is actually descending. Measured on raw-scale Turbidity, an MAE criterion
    stops at **epoch 1** while MSE stops at 114: squared-error training moves the
    prediction off the median and toward the mean, so on a right-skewed target
    validation MAE rises from the very first epoch even while the fit improves.
    Stopping there returns an untrained network — R2 0.007 — and, worse, it
    corrupts the transform bake-off downstream, which then compares two equally
    untrained arms and cannot see the log arm's advantage.

    MAE remains the arbiter *between* raw and log arms in
    `select_target_transform`, where it is scale-neutral and the objectives
    differ. Within one arm the objective is fixed, so the training loss is the
    right thing to watch.
    """
    splitter = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_SIZE, random_state=seed)
    fit_idx, val_idx = next(splitter.split(X, groups=groups))
    if len(val_idx) == 0 or len(fit_idx) == 0:      # degenerate group structure
        return MAX_EPOCHS // 3, float("nan")

    pre = make_preprocessor().fit(X[fit_idx])
    X_fit, X_val = pre.transform(X[fit_idx]), pre.transform(X[val_idx])
    y_fit, y_val = y[fit_idx], y[val_idx]

    # max_iter=1 + warm_start => one further epoch per .fit() call.
    net = MLPRegressor(**MLP_PARAMS, max_iter=1, warm_start=True, random_state=seed)

    best_mse, best_epoch, stalled = np.inf, 1, 0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        for epoch in range(1, MAX_EPOCHS + 1):
            net.fit(X_fit, y_fit)
            mse = mean_squared_error(y_val, net.predict(X_val))
            if mse < best_mse:
                best_mse, best_epoch, stalled = mse, epoch, 0
            else:
                stalled += 1
                if stalled >= PATIENCE:
                    break
    return best_epoch, float(best_mse)


def eligible_for_log(y_train: np.ndarray) -> bool:
    """Whether this target may enter the raw-vs-log bake-off at all."""
    if y_train.min() < 0:
        return False
    return float((y_train == 0).mean()) < MAX_ZERO_FRACTION_FOR_LOG


def fit_arm(X: np.ndarray, y: np.ndarray, groups: np.ndarray, offset: float | None,
            seeds: tuple[int, ...] = SEEDS):
    """Fit one arm of the bake-off. Returns (pipeline, smearing factor, epochs).

    The epoch budget is searched on the transformed target, because raw and log
    scales do not converge at the same rate — handing both arms one shared
    budget would decide the bake-off on whichever scale that budget happened to
    suit.
    """
    y_fit = to_log(y, offset)
    epochs, _ = search_epochs(X, y_fit, groups)
    pipeline = make_pipeline(epochs, seeds)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        pipeline.fit(X, y_fit)
    if offset is None:
        return pipeline, 1.0, epochs
    return pipeline, duan_smearing(y_fit - pipeline.predict(X)), epochs


def select_target_transform(X_train: np.ndarray, y_train: np.ndarray,
                            groups_train: np.ndarray):
    """Choose raw vs log10(y + c) for one target, on training stations only.

    Both arms are cross-validated over the training stations with `GroupKFold`,
    and scored as **MAE in the target's own units**. MAE is the yardstick for
    two reasons: it is the error the dashboard actually displays, and it is the
    only candidate that does not structurally favour one arm — raw-scale R2
    always flatters the raw fit and log-scale R2 the log fit, so neither can
    arbitrate between them.

    The comparison never touches the test set. Choosing on test rows and then
    reporting test scores for the winner would inflate every number in Part 2.

    Each fold is fitted with a single seed rather than the full three-seed
    ensemble: the arms are compared under identical rules, so the extra members
    would cost three times the runtime without changing which arm wins.

    Returns (offset, cv_mae_raw, cv_mae_log). `offset is None` means the raw
    arm won, or the target never qualified — in which case both MAEs are NaN.
    """
    if not eligible_for_log(y_train):
        return None, np.nan, np.nan

    one_seed = (RANDOM_STATE,)
    mae_raw, mae_log = [], []
    for fit_idx, score_idx in GroupKFold(n_splits=N_SELECTION_FOLDS).split(
            X_train, groups=groups_train):
        X_fit, y_fit, g_fit = X_train[fit_idx], y_train[fit_idx], groups_train[fit_idx]
        X_score, y_score = X_train[score_idx], y_train[score_idx]

        offset = log_offset(y_fit)
        raw_pipe, _, _ = fit_arm(X_fit, y_fit, g_fit, None, one_seed)
        log_pipe, smear, _ = fit_arm(X_fit, y_fit, g_fit, offset, one_seed)

        mae_raw.append(mean_absolute_error(y_score, raw_pipe.predict(X_score)))
        mae_log.append(mean_absolute_error(
            y_score, to_raw(log_pipe.predict(X_score), offset, smear)))

    cv_raw, cv_log = float(np.mean(mae_raw)), float(np.mean(mae_log))
    keep_log = cv_log < cv_raw * (1.0 - MIN_LOG_MAE_GAIN)
    return (log_offset(y_train) if keep_log else None), cv_raw, cv_log


def prepare_frame(df: pd.DataFrame, column: str, valid_range: tuple) -> pd.DataFrame:
    """Rows usable for one target: predictors + target + station id + timestamp.

    Drops rows with no measurement and clips to the plausible range. The station
    id and timestamp ride along because the split is grouped by station and the
    persistence baseline needs each station's observations in time order.
    """
    frame = (
        df[FEATURE_COLS + [column, GROUP_COL, "_obs_ts"]]
        .dropna(subset=[column, GROUP_COL])
        .copy()
    )
    lo, hi = valid_range
    return frame[frame[column].between(lo, hi)]


def grouped_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Hold out whole stations — no station may straddle the train/test boundary."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(frame, groups=frame[GROUP_COL]))
    return frame.iloc[train_idx], frame.iloc[test_idx]


TRAINED: dict[str, dict] = {}

for label, cfg in TARGETS.items():
    column = cfg["column"]
    frame = prepare_frame(data, column, cfg["valid_range"])
    if len(frame) < MIN_SAMPLES:
        print(f"[SKIP] {label}: only {len(frame)} usable rows (< {MIN_SAMPLES}).")
        continue

    train, test = grouped_split(frame)
    train_stations = set(train[GROUP_COL])
    test_stations = set(test[GROUP_COL])
    assert not (train_stations & test_stations), f"{label}: station leaked across the split"

    X_train = train[FEATURE_COLS].to_numpy(dtype=float)
    y_train = train[column].to_numpy(dtype=float)
    X_test = test[FEATURE_COLS].to_numpy(dtype=float)
    y_test = test[column].to_numpy(dtype=float)
    g_train = train[GROUP_COL].to_numpy()

    # Pick the target scale using training stations only, then refit the
    # winning arm — as the full seed ensemble — on all of them.
    offset, cv_mae_raw, cv_mae_log = select_target_transform(X_train, y_train, g_train)
    pipeline, smear, epochs = fit_arm(X_train, y_train, g_train, offset)

    TRAINED[label] = {
        "column": column,
        "model": pipeline,
        "offset": offset,           # None => this model was fitted raw
        "smear": smear,             # Duan correction; 1.0 when fitted raw
        "epochs": epochs,           # budget chosen by the grouped search
        "cv_mae_raw": cv_mae_raw,   # the bake-off evidence, for the metrics table
        "cv_mae_log": cv_mae_log,
        "test": test,               # full frame — the persistence baseline needs it
        "X_test": X_test,
        "y_test": y_test,
        "n_total": len(frame),
        "n_train": len(train),
        "n_test": len(test),
        "n_train_stations": len(train_stations),
        "n_test_stations": len(test_stations),
    }
    # Hitting MAX_EPOCHS means validation MSE was still improving when the search
    # ran out of room: the budget was truncated rather than chosen, so this
    # model is trained to the ceiling and not to its own optimum.
    at_ceiling = epochs >= MAX_EPOCHS
    print(f"[OK]   {label:24s} train {len(train):>6,} rows / {len(train_stations):>4,} stations"
          f"  |  test {len(test):>5,} rows / {len(test_stations):>4,} stations"
          f"  ({len(test) / len(frame):>4.0%} of rows)"
          f"  | epochs {epochs:>3d}{'*' if at_ceiling else ' '}"
          + (f"  | log10(y + {offset:.4g}) smear {smear:.3f}" if offset else ""))
    if at_ceiling:
        print(f"       * hit MAX_EPOCHS ({MAX_EPOCHS}) — validation MSE was still "
              "falling; this budget is a ceiling, not an optimum.")

print(f"\nTrained {len(TRAINED)} of {len(TARGETS)} target models "
      f"({len(SEEDS)} seeds each). "
      "No station appears in both the training and the test set.")

[OK]   Water Temperature        train 28,613 rows /  799 stations  |  test 6,092 rows /  200 stations  ( 18% of rows)  | epochs  24 


[OK]   Dissolved Oxygen         train 25,185 rows /  727 stations  |  test 6,640 rows /  182 stations  ( 21% of rows)  | epochs  10 


[OK]   pH                       train 26,680 rows /  885 stations  |  test 5,673 rows /  222 stations  ( 18% of rows)  | epochs  14 


[OK]   Nitrate                  train  9,628 rows /  222 stations  |  test 2,719 rows /   56 stations  ( 22% of rows)  | epochs   9 


[OK]   Nitrite                  train  9,340 rows /  183 stations  |  test 2,271 rows /   46 stations  ( 20% of rows)  | epochs  25 


...

[OK]   Total Dissolved Solids   train 14,109 rows /  465 stations  |  test 3,902 rows /  117 stations  ( 22% of rows)  | epochs  40 


[OK]   Total Suspended Solids   train 12,088 rows /  442 stations  |  test 2,435 rows /  111 stations  ( 17% of rows)  | epochs  18 


[OK]   Turbidity                train 17,575 rows /  675 stations  |  test 3,581 rows /  169 stations  ( 17% of rows)  | epochs   8   | log10(y + 0.14) smear 1.693


[OK]   E. coli                  train 12,066 rows /  347 stations  |  test 3,831 rows /   87 stations  ( 24% of rows)  | epochs 300*
       * hit MAX_EPOCHS (300) — validation MSE was still falling; this budget is a ceiling, not an optimum.


[OK]   WQI                      train 22,676 rows /  754 stations  |  test 4,622 rows /  189 stations  ( 17% of rows)  | epochs  15 

Trained 13 of 13 target models (3 seeds each). No station appears in both the training and the test set.


## Part 2 — Test every model

`test_model()` scores one cached model on its held-out test set and returns the
metrics. `persistence_baseline()` scores the zero-feature alternative — "repeat
this station's previous value" — on the same held-out rows, so every R² can be
read against the memorisation bar it has to clear. `test_all_models()` runs both
across every trained target and assembles a summary table.

In [7]:
def test_model(label: str, verbose: bool = True) -> dict:
    """Evaluate one trained model on its held-out test set.

    Every metric is reported in the target's own units, so the thirteen stay
    comparable and `app.py` keeps reading one scale. For the log-fitted targets
    an `R2 (log)` is reported beside it, because the raw-scale R2 of a log fit
    is dominated by the same extreme tail the transform exists to de-emphasise
    and on its own it understates the model badly. The two answer different
    questions -- "how close in mg/L" and "how close in order of magnitude".
    """
    if label not in TRAINED:
        raise KeyError(f"No trained model for {label!r}. Run Part 1 first.")
    entry = TRAINED[label]
    offset = entry["offset"]
    y_pred = predict_raw(entry, entry["X_test"])
    metrics = evaluate(entry["y_test"], y_pred)
    metrics["R2 (log)"] = np.nan if offset is None else float(r2_score(
        to_log(entry["y_test"], offset),
        entry["model"].predict(entry["X_test"]),
    ))
    if verbose:
        print(f"{label}  (n_test={entry['n_test']:,} rows "
              f"from {entry['n_test_stations']:,} unseen stations)")
        print(f"    R2         = {metrics['R2']:.4f}")
        if offset is not None:
            print(f"    R2 (log)   = {metrics['R2 (log)']:.4f}"
                  "   <- the scale this target is fitted and read on")
        print(f"    RMSE       = {metrics['RMSE']:.4f}")
        print(f"    MAE        = {metrics['MAE']:.4f}")
        print(f"    Error Rate = {metrics['Error Rate (%)']:.2f}%")
    return metrics


def persistence_baseline(label: str) -> dict:
    """Score "repeat this station's previous value" on the same held-out rows.

    Because the split is grouped by station, every observation a test station
    ever made sits in the test set — so this baseline is free to use the site's
    own history, which the model never saw. That makes it the memorisation bar:
    a model that cannot beat it is recalling the site rather than predicting the
    water. (Read the margin with the revisit gap in mind — a target resampled
    the next day is far easier to persist than one resampled a month later.)

    Both scores are computed on the *same* rows — those that have a previous
    observation at the same station — so the margin is like-for-like.

    For a log-fitted target the comparison is run on both scales. Scoring a log
    fit against persistence in raw units handicaps it on exactly the tail the
    transform was chosen to down-weight, so `Margin (log)` is the like-for-like
    number there.
    """
    entry = TRAINED[label]
    column = entry["column"]
    offset = entry["offset"]
    ordered = entry["test"].sort_values([GROUP_COL, "_obs_ts"])
    by_station = ordered.groupby(GROUP_COL, observed=True)
    prev = by_station[column].shift(1).to_numpy(dtype=float)
    gap_days = by_station["_obs_ts"].diff().dt.days.to_numpy(dtype=float)
    paired = ~np.isnan(prev)

    out = {
        "n_pairs": int(paired.sum()),
        "median_gap_days": np.nan,
        "Persistence R2": np.nan,
        "Persistence RMSE": np.nan,
        "Model R2 (paired)": np.nan,
        "Model RMSE (paired)": np.nan,
        "Margin": np.nan,
        "Persistence R2 (log)": np.nan,
        "Model R2 paired (log)": np.nan,
        "Margin (log)": np.nan,
    }
    if out["n_pairs"] < MIN_PERSISTENCE_PAIRS:
        return out

    rows = ordered[paired]
    X_paired = rows[FEATURE_COLS].to_numpy(dtype=float)
    y_true = rows[column].to_numpy(dtype=float)
    y_prev = prev[paired]
    y_pred = predict_raw(entry, X_paired)

    out["median_gap_days"] = float(np.median(gap_days[paired]))
    out["Persistence R2"] = float(r2_score(y_true, y_prev))
    out["Persistence RMSE"] = float(mean_squared_error(y_true, y_prev) ** 0.5)
    out["Model R2 (paired)"] = float(r2_score(y_true, y_pred))
    out["Model RMSE (paired)"] = float(mean_squared_error(y_true, y_pred) ** 0.5)
    out["Margin"] = out["Model R2 (paired)"] - out["Persistence R2"]

    if offset is not None:
        y_true_log = to_log(y_true, offset)
        out["Persistence R2 (log)"] = float(r2_score(y_true_log, to_log(y_prev, offset)))
        out["Model R2 paired (log)"] = float(
            r2_score(y_true_log, entry["model"].predict(X_paired)))
        out["Margin (log)"] = out["Model R2 paired (log)"] - out["Persistence R2 (log)"]
    return out


def test_all_models() -> pd.DataFrame:
    rows = []
    for label, entry in TRAINED.items():
        m = test_model(label, verbose=False)
        p = persistence_baseline(label)
        rows.append({
            "Target": label,
            "Column": entry["column"],
            "N test": entry["n_test"],
            "N test stations": entry["n_test_stations"],
            "R2": m["R2"],
            "R2 (log)": m["R2 (log)"],
            "RMSE": m["RMSE"],
            "MAE": m["MAE"],
            "Error Rate (%)": m["Error Rate (%)"],
            "Persistence R2": p["Persistence R2"],
            "Model R2 (paired)": p["Model R2 (paired)"],
            "Margin": p["Margin"],
            "Margin (log)": p["Margin (log)"],
        })
    summary = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
    return summary

### Per-model report

In [8]:
for label in TRAINED:
    test_model(label)
    print()

Water Temperature  (n_test=6,092 rows from 200 unseen stations)
    R2         = 0.9207
    RMSE       = 2.4827
    MAE        = 1.8637
    Error Rate = 25.79%

Dissolved Oxygen  (n_test=6,640 rows from 182 unseen stations)
    R2         = 0.4520
    RMSE       = 1.9885
    MAE        = 1.4163
    Error Rate = 16.68%

pH  (n_test=5,673 rows from 222 unseen stations)
    R2         = 0.1621
    RMSE       = 0.5799
    MAE        = 0.4372
    Error Rate = 5.55%

Nitrate  (n_test=2,719 rows from 56 unseen stations)
    R2         = 0.2382
    RMSE       = 4.9987
    MAE        = 3.1630
    Error Rate = 111.05%

Nitrite  (n_test=2,271 rows from 46 unseen stations)
    R2         = -0.1864
    RMSE       = 0.1334
    MAE        = 0.0564
    Error Rate = 186.91%

Nitrate + Nitrite  (n_test=1,045 rows from 74 unseen stations)
    R2         = 0.3261
    RMSE       = 3.7106
    MAE        = 2.6391
    Error Rate = 74.99%

Total Phosphorus  (n_test=1,294 rows from 91 unseen stations)
    R2   

### Summary table

All thirteen models side by side, sorted by test R².

In [9]:
summary = test_all_models()
summary

,Target,Column,N test,N test stations,R2,R2 (log),RMSE,MAE,Error Rate (%),Persistence R2,Model R2 (paired),Margin,Margin (log)
0,Water Temperature,"Temperature, water_value",6092,200,0.9207,NaN,2.4827,1.8637,25.7909,0.6403,0.9229,0.2825,NaN
1,Dissolved Oxygen,Dissolved oxygen (DO)_value,6640,182,0.4520,NaN,1.9885,1.4163,16.6779,0.3249,0.4636,0.1387,NaN
2,Total Dissolved Solids,Total dissolved solids_value,3902,117,0.4249,NaN,95.3661,75.8143,24.6800,0.8107,0.4236,-0.3871,NaN
3,Nitrate + Nitrite,Nitrate + Nitrite_value,1045,74,0.3261,NaN,3.7106,2.6391,74.9922,0.1896,0.2859,0.0963,NaN
4,Nitrate,Nitrate_value,2719,56,0.2382,NaN,4.9987,3.1630,111.0477,0.3984,0.2345,-0.1639,NaN
5,pH,pH_value,5673,222,0.1621,NaN,0.5799,0.4372,5.5510,-0.0605,0.1652,0.2257,NaN
6,WQI,WQI,4622,189,0.1056,NaN,16.2235,13.2849,32.5169,0.1718,0.1101,-0.0617,NaN
7,Total Suspended Solids,Total suspended solids_value,2435,111,0.0652,NaN,201.6583,72.7495,104.9688,-0.9931,0.0514,1.0445,NaN
8,Turbidity,Turbidity_value,3581,169,0.0575,0.2245,85.0123,26.1877,79.7922,-0.7883,0.0590,0.8473,0.1770
9,E. coli,Escherichia coli_value,3831,87,0.0256,NaN,"9,954.7329","2,273.3008",132.1712,-0.7406,0.0262,0.7667,NaN


### Persistence baseline — how much of the score is memorisation?

A model with **no features at all** — "this station's next value equals its
previous value" — is the bar any of these models has to clear before the word
*prediction* applies. `Margin` is the model's R² minus the persistence R² on the
identical set of rows; a negative margin means 29 environmental predictors buy
less than repeating the last reading.

In [10]:
persistence = (
    pd.DataFrame([{"Target": label, **persistence_baseline(label)} for label in TRAINED])
    .set_index("Target")
    .sort_values("Margin")
)

view = persistence[["n_pairs", "median_gap_days", "Persistence R2",
                    "Persistence RMSE", "Model R2 (paired)", "Model RMSE (paired)",
                    "Margin",
                    # NaN except on the log-fitted targets, where this is
                    # the like-for-like comparison.
                    "Persistence R2 (log)", "Model R2 paired (log)", "Margin (log)"]]
print(view.round(3).to_string())

beaten = persistence["Margin"] > 0
print(f"\nBeats persistence on {int(beaten.sum())} of {int(beaten.notna().sum())} "
      "targets with a reportable baseline.")
view

                        n_pairs  median_gap_days  Persistence R2  Persistence RMSE  Model R2 (paired)  Model RMSE (paired)  Margin  Persistence R2 (log)  Model R2 paired (log)  Margin (log)
Target                                                                                                                                                                                       
Specific Conductance       4520           1.0000          0.8560           75.4690             0.0120             197.6220 -0.8440                   NaN                    NaN           NaN
Total Dissolved Solids     3785          28.0000          0.8110           54.5530             0.4240              95.1850 -0.3870                   NaN                    NaN           NaN
Total Phosphorus           1203          33.0000          0.3190            0.3330             0.0160               0.4000 -0.3030                0.3770                 0.0930       -0.2840
Nitrate                    2663          15.0000  

,n_pairs,median_gap_days,Persistence R2,Persistence RMSE,Model R2 (paired),Model RMSE (paired),Margin,Persistence R2 (log),Model R2 paired (log),Margin (log)
Target,,,,,,,,,,
Specific Conductance,4520,1.0000,0.8559,75.4694,0.0119,197.6224,-0.8440,NaN,NaN,NaN
Total Dissolved Solids,3785,28.0000,0.8107,54.5535,0.4236,95.1849,-0.3871,NaN,NaN,NaN
Total Phosphorus,1203,33.0000,0.3191,0.3326,0.0160,0.3999,-0.3032,0.3771,0.0931,-0.2840
Nitrate,2663,15.0000,0.3984,4.4065,0.2345,4.9705,-0.1639,NaN,NaN,NaN
WQI,4433,27.0000,0.1718,15.5667,0.1101,16.1361,-0.0617,NaN,NaN,NaN
Nitrate + Nitrite,971,33.0000,0.1896,3.9838,0.2859,3.7396,0.0963,NaN,NaN,NaN
Dissolved Oxygen,6458,20.0000,0.3249,2.1921,0.4636,1.9539,0.1387,NaN,NaN,NaN
pH,5451,28.0000,-0.0605,0.6540,0.1652,0.5803,0.2257,NaN,NaN,NaN
Water Temperature,5892,27.0000,0.6403,5.3114,0.9229,2.4599,0.2825,NaN,NaN,NaN


### Test a single model on demand

Change `target` to re-run the evaluation for any one model.

In [11]:
target = "Water Temperature"
_ = test_model(target)

Water Temperature  (n_test=6,092 rows from 200 unseen stations)
    R2         = 0.9207
    RMSE       = 2.4827
    MAE        = 1.8637
    Error Rate = 25.79%


## Part 3 — Seed stability

A random forest has `feature_importances_`; an MLP has no comparable built-in,
so this section reports something the tree notebooks cannot and this family
needs: **how much of each score is the random initialisation.**

Every saved model averages three networks that differ *only* by seed. Because
`VotingRegressor` keeps its fitted members in `.estimators_`, each one can be
scored separately on the same held-out rows. The spread between them is a
lower bound on how much any single-seed number should be trusted.

Read `Spread` against the margins in the summary table above. Where the spread
is of the same size as the gap between this family and another, the two are
tied and the ranking between them is noise.

In [12]:
def seed_stability(label: str) -> dict:
    """Per-seed test R2 for one model, plus the ensemble's own score.

    Each member is scored through the same back-transform as the ensemble, so
    every number is a raw-unit R2 on the identical test rows.
    """
    entry = TRAINED[label]
    estimator = entry["model"].named_steps["model"]
    if not hasattr(estimator, "estimators_"):       # single-seed fallback
        return {}

    # Re-use the ensemble's own fitted imputer+scaler; members were fitted on it.
    pre = Pipeline(entry["model"].steps[:-1])
    X = pre.transform(entry["X_test"])
    y_true = entry["y_test"]

    scores = [
        float(r2_score(y_true, to_raw(member.predict(X), entry["offset"], entry["smear"])))
        for member in estimator.estimators_
    ]
    return {
        "Seeds": len(scores),
        "Min R2": min(scores),
        "Max R2": max(scores),
        "Mean R2": float(np.mean(scores)),
        "SD R2": float(np.std(scores, ddof=1)) if len(scores) > 1 else np.nan,
        "Spread": max(scores) - min(scores),
        "Ensemble R2": float(r2_score(y_true, predict_raw(entry, entry["X_test"]))),
    }


stability = (
    pd.DataFrame([{"Target": label, **seed_stability(label)} for label in TRAINED])
    .set_index("Target")
    .sort_values("Spread", ascending=False)
)
print(stability.round(4).to_string())

gain = (stability["Ensemble R2"] - stability["Mean R2"]).mean()
print(f"\nAveraging {len(SEEDS)} seeds is worth {gain:+.4f} R2 on average "
      "versus a single network.")
stability

                        Seeds  Min R2  Max R2  Mean R2  SD R2  Spread  Ensemble R2
Target                                                                            
Nitrite                     3 -1.3732 -0.1871  -0.6937 0.6117  1.1862      -0.1864
Turbidity                   3 -0.0349  0.0548   0.0162 0.0461  0.0897       0.0575
pH                          3  0.0626  0.1356   0.1083 0.0399  0.0730       0.1621
Total Phosphorus            3 -0.0692  0.0021  -0.0245 0.0390  0.0713       0.0202
Nitrate                     3  0.2015  0.2727   0.2304 0.0375  0.0713       0.2382
Nitrate + Nitrite           3  0.2832  0.3293   0.3101 0.0239  0.0460       0.3261
Specific Conductance        3 -0.0016  0.0349   0.0163 0.0182  0.0364       0.0203
WQI                         3  0.0854  0.1162   0.1007 0.0154  0.0308       0.1056
Dissolved Oxygen            3  0.4249  0.4551   0.4420 0.0155  0.0302       0.4520
Total Dissolved Solids      3  0.4081  0.4280   0.4199 0.0104  0.0199       0.4249
Tota

,Seeds,Min R2,Max R2,Mean R2,SD R2,Spread,Ensemble R2
Target,,,,,,,
Nitrite,3,-1.3732,-0.1871,-0.6937,0.6117,1.1862,-0.1864
Turbidity,3,-0.0349,0.0548,0.0162,0.0461,0.0897,0.0575
pH,3,0.0626,0.1356,0.1083,0.0399,0.0730,0.1621
Total Phosphorus,3,-0.0692,0.0021,-0.0245,0.0390,0.0713,0.0202
Nitrate,3,0.2015,0.2727,0.2304,0.0375,0.0713,0.2382
Nitrate + Nitrite,3,0.2832,0.3293,0.3101,0.0239,0.0460,0.3261
Specific Conductance,3,-0.0016,0.0349,0.0163,0.0182,0.0364,0.0203
WQI,3,0.0854,0.1162,0.1007,0.0154,0.0308,0.1056
Dissolved Oxygen,3,0.4249,0.4551,0.4420,0.0155,0.0302,0.4520


### Permutation importance

The nearest available substitute for the forest's impurity importances: shuffle
one predictor's column in the test set and measure how far R² falls. It is
computed on **raw units through the back-transform**, so it is comparable across
targets, and it is model-agnostic — the same routine would run on any family.

Shown for one target by default; change `target` to inspect another. Each
feature is shuffled `n_repeats` times, so cost scales with the feature count.

In [13]:
def permutation_importance_raw(label: str, n_repeats: int = 5,
                               seed: int = RANDOM_STATE) -> pd.DataFrame:
    """Drop in test R2 (raw units) when each predictor is shuffled.

    Hand-rolled rather than sklearn's `permutation_importance` because the score
    has to be taken *after* the log back-transform for the log-fitted targets;
    scoring the pipeline directly would silently mix log-scale and raw-scale
    numbers across the thirteen.
    """
    entry = TRAINED[label]
    X = entry["X_test"].copy()
    y_true = entry["y_test"]
    base = r2_score(y_true, predict_raw(entry, X))

    rng = np.random.default_rng(seed)
    rows = []
    for j, name in enumerate(FEATURE_COLS):
        original = X[:, j].copy()
        drops = []
        for _ in range(n_repeats):
            X[:, j] = rng.permutation(original)
            drops.append(base - r2_score(y_true, predict_raw(entry, X)))
        X[:, j] = original
        rows.append({"feature": name, "r2_drop": float(np.mean(drops)),
                     "sd": float(np.std(drops, ddof=1)) if n_repeats > 1 else np.nan})

    return (pd.DataFrame(rows)
            .sort_values("r2_drop", ascending=False)
            .reset_index(drop=True)
            .assign(baseline_r2=base))


target = "Water Temperature"
permutation_importance_raw(target).head(10)

,feature,r2_drop,sd,baseline_r2
0,doy_cos,0.4597,0.0060,0.9207
1,doy_sin,0.0711,0.0022,0.9207
2,prism_tmax_c,0.0445,0.0012,0.9207
3,prism_tmin_c,0.0429,0.0015,0.9207
4,awc_mean,0.0278,0.0007,0.9207
5,isu_max_feel_c,0.0275,0.0008,0.9207
6,prism_tdmean_c,0.0252,0.0011,0.9207
7,ksat_mean,0.0202,0.0003,0.9207
8,isu_min_feel_c,0.0147,0.0005,0.9207
9,npmanure__total__n_kg,0.0145,0.0004,0.9207


## Part 4 — Save trained models

Persist every fitted pipeline to `src/05_modeling/neural_network/` as
`nn_<target>.pkl`. Each file is self-contained (imputer + scaler + estimator)
and can be reloaded with `pickle.load` for inference.

In [14]:
import pickle
import re

# Write .pkl files into src/05_modeling/neural_network/ regardless of launch dir.
MODEL_DIR = REPO_ROOT / "src" / "05_modeling" / "neural_network"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "nn"  # neural network

def target_stem(label: str) -> str:
    """'Nitrate + Nitrite' -> 'nitrate_nitrite', 'E. coli' -> 'e_coli'."""
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")

# Each .pkl is a plain dict: the fitted pipeline plus everything needed to
# interpret what it predicts. Keeping the transform inside the artifact means
# app.py cannot mislabel a log10 prediction as mg/L when model_metrics.csv is
# stale or missing — the two can no longer drift apart.
#
# Every component here is stock scikit-learn — SimpleImputer, StandardScaler,
# VotingRegressor, MLPRegressor — so the artifact matches the other three
# families' contract exactly and still unpickles with no bespoke class to
# import. That is the specific reason this family is an `MLPRegressor` rather
# than a torch module: a saved `nn.Module` needs its class definition importable
# at load time, which is the fragility that got model_feature_engineering.py
# deleted.
saved = []
for label, entry in TRAINED.items():
    path = MODEL_DIR / f"{PREFIX}_{target_stem(label)}.pkl"
    artifact = {
        "pipeline": entry["model"],
        "target_transform": "log10" if entry["offset"] is not None else "none",
        "log_offset": entry["offset"],
        "smearing_factor": entry["smear"],
        "feature_cols": list(FEATURE_COLS),
    }
    with open(path, "wb") as f:
        pickle.dump(artifact, f)
    saved.append(path.name)

print(f"Saved {len(saved)} models to {MODEL_DIR}:")
for name in saved:
    print("  ", name)

Saved 13 models to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/src/05_modeling/neural_network:
   nn_water_temperature.pkl
   nn_dissolved_oxygen.pkl
   nn_ph.pkl
   nn_nitrate.pkl
   nn_nitrite.pkl
   nn_nitrate_nitrite.pkl
   nn_total_phosphorus.pkl
   nn_specific_conductance.pkl
   nn_total_dissolved_solids.pkl
   nn_total_suspended_solids.pkl
   nn_turbidity.pkl
   nn_e_coli.pkl
   nn_wqi.pkl


## Part 5 — Publish metrics

Write this family's held-out results — including the station counts and the
persistence baseline — into `src/05_modeling/model_metrics.csv`, the table the
dashboard reads and `model_outcomes.md` summarises.

In [15]:
# --- Publish the metrics table read by app.py and model_outcomes.md ---
# Each notebook owns its own family's rows: we replace them in place and leave
# the other families untouched, so the notebooks can be run in any order.
FAMILY = "Neural Network"
FAMILY_ORDER = ["Linear Regression", "Random Forest", "Gradient Boosting", "Neural Network"]
METRICS_PATH = REPO_ROOT / "src" / "05_modeling" / "model_metrics.csv"

def _round(value, digits: int = 4):
    """Round, tolerating the None/NaN of a raw-scale target."""
    return np.nan if value is None or pd.isna(value) else round(float(value), digits)


rows = []
for label, entry in TRAINED.items():
    m = test_model(label, verbose=False)
    p = persistence_baseline(label)
    rows.append({
        "target": label,
        "model": FAMILY,
        "r2": _round(m["R2"]),
        "rmse": _round(m["RMSE"]),
        "mae": _round(m["MAE"]),
        "error_rate": _round(m["Error Rate (%)"], 2),
        "test_rows": entry["n_test"],
        "test_stations": entry["n_test_stations"],
        "train_rows": entry["n_train"],
        "train_stations": entry["n_train_stations"],
        "persistence_pairs": p["n_pairs"],
        "persistence_median_gap_days": p["median_gap_days"],
        "persistence_r2": _round(p["Persistence R2"]),
        "model_r2_paired": _round(p["Model R2 (paired)"]),
        "model_minus_persistence": _round(p["Margin"]),
        # --- log10(y + c) targets; NaN on those fitted raw -------------------
        "target_transform": "log10" if entry["offset"] is not None else "none",
        "log_offset": _round(entry["offset"], 8),
        "smearing_factor": _round(entry["smear"] if entry["offset"] else None, 6),
        # The bake-off evidence: mean CV MAE of each arm over the training
        # stations. NaN when the target never qualified for the log arm.
        "cv_mae_raw_fit": _round(entry["cv_mae_raw"], 6),
        "cv_mae_log_fit": _round(entry["cv_mae_log"], 6),
        "r2_log": _round(m["R2 (log)"]),
        "persistence_r2_log": _round(p["Persistence R2 (log)"]),
        "model_r2_paired_log": _round(p["Model R2 paired (log)"]),
        "model_minus_persistence_log": _round(p["Margin (log)"]),
    })

family_metrics = pd.DataFrame(rows)

if METRICS_PATH.exists():
    existing = pd.read_csv(METRICS_PATH)
    if set(existing.columns) != set(family_metrics.columns):
        print("[WARN] existing model_metrics.csv uses the older schema — dropping its "
              "rows. Re-run the other notebooks to refill them.")
        # Take the empty frame from *family_metrics*, not from `existing`:
        # `existing.iloc[0:0]` drops the rows but keeps the stale column names,
        # and the concat below would union them straight back in.
        existing = family_metrics.iloc[0:0]
    combined = pd.concat([existing[existing["model"] != FAMILY], family_metrics],
                         ignore_index=True)
else:
    combined = family_metrics

combined = (
    combined
    .assign(_f=pd.Categorical(combined["model"], FAMILY_ORDER, ordered=True),
            _t=pd.Categorical(combined["target"], list(TARGETS), ordered=True))
    .sort_values(["_f", "_t"])
    .drop(columns=["_f", "_t"])
)
combined.to_csv(METRICS_PATH, index=False)
print(f"Wrote {len(family_metrics)} {FAMILY} rows "
      f"({len(combined)} total) to {METRICS_PATH}")
family_metrics

Wrote 13 Neural Network rows (52 total) to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/src/05_modeling/model_metrics.csv


,target,model,r2,rmse,mae,error_rate,test_rows,test_stations,train_rows,train_stations,...,model_minus_persistence,target_transform,log_offset,smearing_factor,cv_mae_raw_fit,cv_mae_log_fit,r2_log,persistence_r2_log,model_r2_paired_log,model_minus_persistence_log
0,Water Temperature,Neural Network,0.9207,2.4827,1.8637,25.7900,6092,200,28613,799,...,0.2825,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dissolved Oxygen,Neural Network,0.4520,1.9885,1.4163,16.6800,6640,182,25185,727,...,0.1387,none,NaN,NaN,1.6827,2.1778,NaN,NaN,NaN,NaN
2,pH,Neural Network,0.1621,0.5799,0.4372,5.5500,5673,222,26680,885,...,0.2257,none,NaN,NaN,0.4707,0.5690,NaN,NaN,NaN,NaN
3,Nitrate,Neural Network,0.2382,4.9987,3.1630,111.0500,2719,56,9628,222,...,-0.1639,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Nitrite,Neural Network,-0.1864,0.1334,0.0564,186.9100,2271,46,9340,183,...,0.5135,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Nitrate + Nitrite,Neural Network,0.3261,3.7106,2.6391,74.9900,1045,74,3614,293,...,0.0963,none,NaN,NaN,2.6440,3.0955,NaN,NaN,NaN,NaN
6,Total Phosphorus,Neural Network,0.0202,0.4720,0.1875,63.9000,1294,91,4592,362,...,-0.3032,log10,0.0012,1.4277,0.2209,0.1723,0.0987,0.3771,0.0931,-0.2840
7,Specific Conductance,Neural Network,0.0203,201.0129,134.1641,26.0800,4617,97,11450,386,...,-0.8440,none,NaN,NaN,132.6661,235.6177,NaN,NaN,NaN,NaN
8,Total Dissolved Solids,Neural Network,0.4249,95.3661,75.8143,24.6800,3902,117,14109,465,...,-0.3871,none,NaN,NaN,76.7696,126.5382,NaN,NaN,NaN,NaN
9,Total Suspended Solids,Neural Network,0.0652,201.6583,72.7495,104.9700,2435,111,12088,442,...,1.0445,none,NaN,NaN,98.7766,110.8286,NaN,NaN,NaN,NaN


---

**Notes**

* This family exists to answer one question: *does a neural network help on this
  feature set at all?* It is the cheap version of that experiment — stock
  scikit-learn, no new dependency, and the same `Pipeline` contract the other
  three families already satisfy.

* **The measured answer is no.** The MLP is beaten by Random Forest or Gradient
  Boosting on 12 of 13 targets. Its one nominal win, Turbidity (+0.020 R² over
  GB, and +0.19 on the log scale it is fitted on), is smaller than its own
  seed spread of 0.090 — so even that is a tie, not a win. The worst losses are
  concentrated where a tree can exploit sharp axis-aligned splits on latitude
  and longitude: Specific Conductance falls from 0.655 (RF) to 0.020, and pH,
  Nitrate, Nitrate + Nitrite and WQI each give up 0.19–0.25. A smooth global
  function is simply the wrong inductive bias for a station-keyed target.

  This is the expected result — on tabular data of this size, gradient-boosted
  trees are the standard-bearer — but it is worth having measured rather than
  assumed, and it is the baseline any richer architecture has to beat.

* **Do not read this as "the network needs to be bigger."** The epoch searches
  land at their true validation optima (Water Temperature stops at epoch 24,
  where the best achievable test R² over all 300 epochs is 0.9171 against the
  0.9168 it takes), and the flat targets are flat for every family. The binding
  constraint is signal in the 29 features, not capacity.
* **What a plain MLP cannot do** is the thing that most justifies a network on
  this dataset: share a representation across targets. 36,696 rows carry two or
  more measured targets and 32,699 carry four or more, and the targets are
  physically coupled (TDS↔specific conductance, TSS↔turbidity, the nitrogen
  series). A shared-trunk network with one head per target and a masked loss
  would let the 3,614-row Nitrate + Nitrite head borrow from the 28,613-row
  Water Temperature rows. `MLPRegressor` fits one target at a time, so that
  experiment needs a different tool — this notebook is the baseline it would
  have to beat.
* **Read Part 3 before ranking this family against another.** Where the
  seed spread is as large as the gap to the next family, the two are tied.
* `MLP_PARAMS`, `SEEDS`, `MAX_EPOCHS` and `PATIENCE` at the top control the fit.
  Widening `hidden_layer_sizes` is the least promising knob here — the binding
  constraint on the flat targets is signal, not capacity — while `alpha` is the
  one worth sweeping on the small-n targets.
* To persist a fitted model, `pickle.dump(TRAINED[label]["model"], ...)`; the
  pipeline is self-contained (imputer + scaler + estimator).